# 📓 Semana 8 · Dia 5 — DLT avançado: triggered vs continuous e expectations profundas

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition (triggered) + 🔑 contínuo (trial) |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEP (DLT) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Pipeline avançado com expectations profundas |

---


## 📖 Teoria — Triggered vs Continuous no DLT

| Modo | Comportamento | Uso |
|---|---|---|
| **Triggered** | processa o delta disponível e para | padrão; custo controlado |
| **Continuous** | processa em streaming contínuo (latência baixa) | dados em tempo real; mais custo |

Na Free Edition, o pipeline é **triggered** (1 ativo por tipo). O modo continuous é recurso de conta completa (trial para validar).


## 📖 Teoria — Expectations avançadas

Além dos 3 níveis, o DLT permite:
- `@dlt.expect_all({...})` — várias expectations, mantém tudo
- `@dlt.expect_all_or_drop({...})` — descarta violadas
- `@dlt.expect_all_or_fail({...})` — falha em qualquer violação
- Combinar com funções: `col('receita') > 0`, `isin(...)`, `regexp`


### 💻 Na prática — Expectations combinadas

Aplique expect_all_or_drop com várias regras numa tabela.


In [ ]:
# ===== workspace_file: pipeline_vendas_avancado.py =====
import dlt
from pyspark.sql.functions import col

@dlt.table(comment="Prata com expectations combinadas")
@dlt.expect_all_or_drop({
    "qtd_positiva": "Quantity > 0",
    "preco_positivo": "UnitPrice > 0",
    "cliente_existe": "CustomerID IS NOT NULL",
    "pais_valido": "UPPER(Country) IN ('UNITED KINGDOM', 'BRAZIL', 'GERMANY', 'FRANCE')"})
def vendas_prata():
    return (dlt.read_stream("vendas_bronze"))

@dlt.table(comment="Ouro: receita diária com expect_or_fail (KPI crítico)")
@dlt.expect_or_fail("receita_nao_negativa", "receita >= 0")
def receita_diaria():
    return (dlt.read("vendas_prata")
        .withColumn("receita", col("Quantity") * col("UnitPrice"))
        .groupBy("data_venda")
        .sum("receita"))

### 💻 Na prática — Modo triggered na Free

Na UI do pipeline: **Triggered** (processa o que há e para). Rode e veja as métricas de qualidade. Para validar **continuous**, use o trial pago.


In [ ]:
# Rodando no modo triggered (UI) — observação das métricas
print("""
1. Pipeline > Settings > Triggered
2. Start
3. Aba Quality: veja violações por expectation
4. Tabelas: vendas_prata (drop de violadas), receita_diaria (fail em receita < 0)
""")
print("Na Free, 1 pipeline ativo por tipo — pare ao terminar.")

> 🎯 **Dica de prova**: DEP: triggered (delta + para) vs continuous (streaming contínuo) e as funções expect_all*. Pergunta típica: 'qual expectation mantém linhas e só monitora?' → expect_all.


## 🎯 Exercícios de fixação

**1.** Quando usar continuous em vez de triggered?

**2.** Escreva expect_all_or_fail com 3 regras de negócio do seu projeto.

**3.** Por que o Ouro usa expect_or_fail para KPI crítico?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Continuous

Quando a latência importa (decisões em tempo real) e o custo é aceitável. Para lotes diários, triggered é suficiente e mais barato.

**2.** Expect_all_or_fail

```python
@dlt.expect_all_or_fail({"qtd>0": "Quantity > 0", "preco>0": "UnitPrice > 0", "cliente": "CustomerID IS NOT NULL"})
```

**3.** Fail no KPI

Um KPI errado propaga erro para BI e modelos — melhor parar o pipeline e alertar do que publicar número errado.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*